Based on the tutorials.

In [ ]:
import lsst.afw.image as afwImage

import matplotlib.pyplot as plt

import lsst.afw.display as afwDisplay
afwDisplay.setDefaultBackend("matplotlib")

import numpy as np

from astropy.table import Table

In [ ]:
def show_mask_planes(mask):
    """
    Print all available mask planes with their bit value (2^n) and display color,
    ordered by ascending bit value.

    Parameters
    ----------
    mask : `lsst.afw.image.Mask`
        Mask object.
    """
    plane_dict = mask.getMaskPlaneDict()
    sorted_planes = sorted(plane_dict.items(), key=lambda item: item[1])

    print(f"{'Mask Plane': <20} {'Bit': <5} {'2^n': <6} {'Hex': <10} {'Default Color'}")
    print("-" * 60)

    for name, bit in sorted_planes:
        color = afw_display.getMaskPlaneColor(name)
        value = 2**bit
        hex_value = hex(value)
        print(f"{name: <20} {bit: <5} {value: <6} {hex_value: <8} {color}")

In [ ]:
ls $HOME/shared*/v1.1_ecdfs_i_0/*fits -lth

In [ ]:
filename = "/home/sfu/shared_lagn_injection/v1.1_ecdfs_i_0/injected_visit_2024110800248_4.fits"
image = afwImage.ExposureF.readFits(filename)

In [ ]:
filename = "/home/sfu/shared_lagn_injection/v1.1_ecdfs_i_0/inj_catalog_visit_2024110800248_4.fits"
catalog = Table.read(filename)

In [ ]:
catalog['x', 'y'][2:5]

In [ ]:
ind = 3
width = 60
x_c = catalog['x'][ind]
y_c = catalog['y'][ind]
xmin = x_c - width/2
xmax = x_c + width/2
ymin = y_c - width/2
ymax = y_c + width/2

image_s = image[xmin:xmax, ymin:ymax]

In [ ]:
fig, ax = plt.subplots(1,1)
plt.sca(ax)
afw_display = afwDisplay.Display(frame=fig)
scale = "asinh"
afw_display.scale(scale, "zscale")
#afw_display.image(image)
afw_display.image(image_s)

In [ ]:
#mask = image.getMask()
mask = image_s.getMask()

In [ ]:
show_mask_planes(mask)

In [ ]:
values, counts = np.unique(mask.array, return_counts=True)

print(f"{'Hex': <8} {'Binary': <6} {'Counts': <8} {'Mask Planes': <30}")
print("-" * 60)

mask_plane_list = []

for value, count in zip(values, counts):
    hex_value = hex(value)
    print(f"{hex_value: <8} {value: <6} {count: <8} {mask.interpret(value): <30}")
    mask_planes = mask.interpret(value).split(',')
    for mask_plane in mask_planes:
        if mask_plane=='':
            continue
        else:
            mask_plane_list.append(mask_plane)

mask_plane_set = set(mask_plane_list)

In [ ]:
mask_plane_set

In [ ]:
#afw_display = afwDisplay.Display(frame=1)
#afw_display.setMaskTransparency(100)
#afw_display.setMaskTransparency(0, 'DETECTED')

In [ ]:
#mask.getMaskPlaneDict().items()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 8))
axs = axs.flatten()
count = 0
for i, (name, bit) in enumerate(mask.getMaskPlaneDict().items()):
    if name not in mask_plane_set:
        continue
    print(i, name, bit)
    #fig.add_subplot(5, 5, i + 1)
    axs[count].imshow(mask.array & 2**bit, vmin=0, vmax=1,
               origin='lower', cmap='Greys',
               interpolation='nearest')
    axs[count].set_title(name)
    axs[count].grid(ls=':')
    #axs[i].axis('off')
    count += 1

In [ ]:
33*1.41

In [ ]:
image_s.getWcs()

In [ ]:
output_filename = "./image_s.fits"
image_s.writeFits(output_filename)

In [ ]:
image_s2 = afwImage.ExposureF.readFits(output_filename)

In [ ]:
image_s2.getWcs()

In [ ]:
image_s2.getVariance()

In [ ]:
image_s2.variance

In [ ]:
image.wcs.pixelToSky(10,10)

In [ ]:
image[10:20,10:20].wcs.pixelToSky(10,10,1,1)

In [ ]:
image[10:20,10:20].wcs.getPixelOrigin()

In [ ]:
image[10:20,10:20].getXY0()

In [ ]:
from astropy.wcs import WCS as astropyWCS

In [ ]:
wcs_header = astropyWCS(image.getWcs().getFitsMetadata()).to_header()

In [ ]:
wcs_header.copy()['CRPIX1']

In [ ]:
image_xy0 = image.getXY0()

In [ ]:
cutout = image[10:20,10:20]

In [ ]:
xy0 = cutout.getXY0()
cutout_wcs_header = wcs_header.copy() 
cutout_wcs_header['CRPIX1'] -= (xy0.getX() - image_xy0.getX())
cutout_wcs_header['CRPIX2'] -= (xy0.getY() - image_xy0.getY())

In [ ]:
cutout_wcs_header

In [ ]:
from astropy.io import fits

In [ ]:
hdu = fits.ImageHDU(cutout.image.array)

In [ ]:
hdu.header.update(cutout_wcs_header) 

In [ ]:
hdu.header

In [ ]:
hdu.writeto('test.fits', overwrite=True)

In [ ]:
header = fits.getheader('test.fits', ext=1)
wcs = astropyWCS(header)

In [ ]:
header

In [ ]:
wcs.pixel_to_world(0, 0)

In [ ]:
image.wcs.pixelToSky(10,10)

In [ ]:
0.00002 * 3600 / 0.2